In [ ]:
# import numpy, matplotlib, and timflow
import numpy as np
import matplotlib.pyplot as plt
import timflow.steady as tfs
import timflow.transient as tft

## Dewatering

The groundwater table at a site near a river needs to be lowered.
The water level in the river is constant at 15 m while the groundwater gradient towards the river is initially 0.001. 
The river runs North-South and is approximated here as straight.
The site that needs to be dewatered is 40 m long (East-West) and 20 m wide (North-South)
The distance between the center of the site and the river is approximately 200 m. The aquifer is 20 m thick and has a hydraulic conductivity of $k=20$ m/d; the base of the aquifer is at $z=0$.

In [ ]:
# code to produce the layout of the problem
plt.figure(figsize=(4, 3))
plt.subplot(111, aspect=1, ylim=(-50, 50))
plt.plot([-20, 20, 20, -20, -20], [-10, -10, 10, 10, -10], 'lightgrey')
plt.text(0, 0, 'site', ha='center')
plt.plot([100, 100], [-50, 50], 'dodgerblue', lw=3)
plt.text(90, 0, 'river', rotation=90, color='dodgerblue');

Twenty four wells with a radius of 5 cm are used to lower the heads at the site. The wells are placed 1 m outside the boundary of the site. The twenty four
wells are connected with a header pipe connected to a single pump so that the head inside
all wells is the same but the discharges are potentially different. The total discharge
of the 24 wells combined must be chosen such that the head at the center of the
site (i.e., at $(x,y)=(0,0)$ is at $h=13$ m. 

### Single layer steady model
The locations of the 24 wells and the $x,y$ coordinates of the river segments are loaded from file. The file contains endpoints of 30 river segments from $(x,y)=(100,-4200)$ till $(x,y)=(100,4200)$ with shorter segments near the site. 

In [ ]:
xywell = np.loadtxt("data/xywell.dat")
xyriv = np.loadtxt("data/xyriv.dat") # 10_000 m section of river

We start by creating a steady one-layer model using the `Model3D` class (we are going to make this into a multi-layer model later on).
We store the model in the variable `ml`. 
We add three analytic elements to the model `ml`:
1. A reference point far away to the left at $(x,y)=(-2000,0)$ where the head is fixed to $h=17$ m, so that the gradient of the background flow towards the river is 0.001.
2. A well string with an initial estimate of the total discharge of $Q=1000$ m$^3$/d. The well string is stored in the variable `ws`. 
3. A river string with a fixed head of 15 m

After we create the model, we solve it. 

In [ ]:
ml = tfs.Model3D(kaq=20, z=[15, 0], topboundary="conf")
rf = tfs.Constant(model=ml, xr=-2000, yr=0, hr=17)
ws = tfs.WellString(model=ml, xy=xywell, Qw=1000, rw=0.05)
river = tfs.RiverString(model=ml, xy=xyriv, hls=15)
ml.solve()

The head may now be computed at any point in the aquifer, for example at $(x,y)=(-150, 80)$

In [ ]:
ml.head(-150, 80)

The head inside the well string may be computed as 

In [ ]:
ws.headinside()

and the total discharge as (just to check whether `timflow` did it correctly):

In [ ]:
ws.discharge()

**Question 1:** 
Compute the head at the center of the site. Do we pump enough?

In [ ]:
ml.head(0, 0)

**Question 2:** 
Create a contour plot with the `ml.plots.contour` function in a window where $x$ varies from `xmin=-80` till `xmax=120` and `ymin=-50` till `ymax=50`. Contour heads from 10 through 15 with a contour interval of 0.2 m (use `np.arange` to create an array) by specifying the `levels` keyword (don't worry, it skips the levels that are not present automatically) and set `parallel=True` to use all the cores on your machine. 

In [ ]:
ml.plots.contour(win=[-80, 120, -50, 50], ngr=50, levels=np.arange(10, 15, 0.2), decimals=1, parallel=True);

**Question 3:**
You probably found out that the head at the center of the site is not low enough yet. 
You have two options: You can use trial and error and increase the discharge of the well string until the head at the center of the site is 13 m. 
Or you can modify the model and replace the `WellString` element by the `TargetHeadWellString` element and let `timflow` compute the discharge for you.

In [ ]:
ml = tfs.Model3D(kaq=20, z=[15, 0], topboundary="conf")
rf = tfs.Constant(model=ml, xr=-2000, yr=0, hr=17)
ws = tfs.TargetHeadWellString(model=ml, xy=xywell, rw=0.05, hcp=13, xcp=0, ycp=0, lcp=0)
river = tfs.RiverString(model=ml, xy=xyriv, hls=15)
ml.solve()

**Question 4:** What is the discharge of the well string such that the head at the center of the site is 13 m? And what is the head inside the well string? And the head at the center of the site (just to check)? Can you make a contour plot?

In [ ]:
print(f'discharge of well string: {ws.discharge()[0]:.0f} m3/d')
print(f'head inside well string: {ws.headinside():.2f} m')
print(f"head at 0, 0: {ml.head(0, 0)[0]:.2f} m")

In [ ]:
# head contours
ml.plots.contour(win=[-80, 120, -50, 50], ngr=50, levels=np.arange(10, 15, 0.2), decimals=1, parallel=True);

**Question 5:** Check that the head along the river is indeed simulated accurately as $h=15$ m by plotting the head along the river from $(x,y)=(100,-100)$ till $(x,y)=(100,100)$. Use the `ml.headalongline` function to compute the head along a line and then plot it.

In [ ]:
y = np.linspace(-100, 100, 100)
x = 100 * np.ones(100)
h = ml.headalongline(x, y)
plt.figure(figsize=(4, 3))
plt.plot(h[0], y)
plt.xlabel('head (m)')
plt.ylabel('y along river (m)')
plt.grid()

### Five-layer steady model
In the previous model, all wells were screened over the entire aquifer thickness of 15
m. Here, we divide the aquifer up in 5 aquifer layers and screen the wells in the second layer while the river
cuts through the first layer only (note that the first layer is numbered 0). 

Modify the single-layer model model such that it consists of 5 layers of 3 m thick. Set the vertical anisotropy to 0.2 using the `kzoverkh` keyword. 
Specify that the river is present only in the first layer and that the wells are screened in the second layer using the `layers` keyword. Make sure the discharge of the well screen is such that the head a the center of the site is 13 m.

Create a contour plot of the top model layer and of the layer that contains the well screens. 

In [ ]:
ml = tfs.Model3D(kaq=20, z=np.linspace(15, 0, 6), topboundary="conf", kzoverkh=0.2)
rf = tfs.Constant(model=ml, xr=-2000, yr=0, hr=17, layer=0)
ws = tfs.TargetHeadWellString(model=ml, xy=xywell, rw=0.05, layers=1, hcp=13, xcp=0, ycp=0, lcp=0)
river = tfs.RiverString(model=ml, xy=xyriv, hls=15, layers=0)
ml.solve()

In [ ]:
ml.plots.contour(win=[-40, 40, -30, 30], ngr=100, levels=np.arange(10, 15, 0.2), decimals=1, parallel=True,
                layers=[0, 1]);

**Question 6:** What is the discharge of the well string and what is the head inside the well string?

In [ ]:
print(f'head inside well string: {ws.headinside():.2f} m')
print(f'discharge of well string: {ws.discharge()[1]:.0f} m3/d')

**Question 7:** Does the required discharge of the well screen go up or down when then vertical anisotropy increases?

In [ ]:
ml = tfs.Model3D(kaq=20, z=np.linspace(15, 0, 6), topboundary="conf", kzoverkh=0.1)
rf = tfs.Constant(model=ml, xr=-2000, yr=0, hr=17, layer=0)
ws = tfs.TargetHeadWellString(model=ml, xy=xywell, rw=0.05, layers=1, hcp=13, xcp=0, ycp=0, lcp=0)
river = tfs.RiverString(model=ml, xy=xyriv, hls=15, layers=0)
ml.solve()

In [ ]:
print(f'head inside well string: {ws.headinside():.2f} m')
print(f'discharge of well string: {ws.discharge()[1]:.0f} m3/d')

### Transient model

Change the 5-layer steady model into a 5-layer transient model. 
Make the following changes to the `Model3D`:
1. Use elements from `tft` (timflow transient).
2. Specify the storage coefficient. The top layer is phreatic with a storage coefficient $S=0.2$ and the specific storage of the other layers is $S_s=0.0001$ $\textrm{m}^{-1}$.
3. Specify that the `topboundary` is `"phreatic"`
4. Specify `tmin=10` and `tmax=100` days. This means that althouh pumping starts at $t=0$, we can start computing heads at $t=10$ d.

There are no target head elements in `timflow.transient` yet, so use a discharge of $Q=1500$ $\textrm{m}^3$/d. For the well string, specify the `tsandQ` as `[(0, 1500)]` (starting time and $Q$). For the river string, specify `tsandh` as `"fixed"` as the water level in the river does not change because of the pumping. Use a shorter section of river in the model, from $(x,y)=(100,-600)$ till $(100, 600)$, defined below as `xyriv_tr` in the code cell below (the head doesn't change much beyond that section for the first 100 days). 
Finally, remove the `Constant` element, as it does not exist in `timflow.transient`. The transient model simulates the changes caused by the transient features in the aquifer, in this case the well string, starting with a zero change at $t=0$.

In [ ]:
xyriv_tr = xyriv[9:22] # river segments to be used in transient model

In [ ]:
ml = tft.Model3D(
    kaq=20,
    z=np.linspace(15, 0, 6),
    Saq=[0.2] + 4 * [1e-4],
    topboundary="phreatic",
    kzoverkh=0.2,
    tmin=10,
    tmax=100,
)
ws = tft.WellString(model=ml, xy=xywell, rw=0.05, tsandQ=[(0, 1500)], layers=1)
river = tft.RiverString(model=ml, xy=xyriv_tr, tsandh="fixed", layers=0)
ml.solve()

**Question 8**
Compute the head at the center of the site in the top layer for $t=10$ d and for $t=100$ d.

In [ ]:
print(f'head at t=10: {ml.head(0, 0, 10, layers=0)[0, 0]:.2f} m')
print(f'head at t=100: {ml.head(0, 0, 100, layers=0)[0, 0]:.2f} m')

**Question 9** To determine when a head change of 2 m is reached at the center of the site, compute and plot the head at the center of the site for time varying from $t=10$ till $t=100$ d. 

In [ ]:
t = np.linspace(10, 100, 100)
h = ml.head(0, 0, t)
plt.figure(figsize=(4, 3))
plt.plot(t, h[0])
plt.xlabel('time (d)')
plt.ylabel('head change (m)')
plt.grid()

**Question 10** Create a contour plot for $x$ varying from -400 till 140 and $y$ varying from -300 till 300 for $t=20$ d after pumping started.

In [ ]:
ml.plots.contour(
    win=[-400, 140, -300, 300], 
    ngr=50, 
    levels=np.arange(-3, 0, 0.2), 
    t=20, 
    layers=0, 
    decimals=1, 
    parallel=True
)
plt.plot([-20, 20, 20, -20, -20], [-10, -10, 10, 10, -10], 'lightgrey')  # outline of site

In [ ]:
# generation of xy data for the river
# yriv = np.hstack((np.linspace(-4200, -200, 11), np.linspace(-200, 200, 11)[1:], np.linspace(200, 4200, 11)[1:]))
# xyriv = np.zeros((len(yriv), 2))
# xyriv[:, 0] = 100
# xyriv[:, 1] = yriv
# np.savetxt('xyriv.dat', xyriv)